In [9]:
from langgraph.graph import StateGraph,END,START
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict,Annotated
from langgraph.graph.message import add_messages
from langchain_core.messages import AnyMessage
from langchain_openai import ChatOpenAI

class ChatState(TypedDict):
    response:Annotated[list[AnyMessage],add_messages]

llm=ChatOpenAI(model='gpt-4o-mini')

def node(state:ChatState)->dict:
    message=state["response"]
    response=llm.invoke(message)
    return {"response":[response]}

graph=StateGraph(ChatState)

graph.add_node("node",node)
graph.add_edge(START,"node")
graph.add_edge("node",END)

memory=MemorySaver()
chatbot=graph.compile(checkpointer=memory)

print(f"Chahtbot with memory created")

Chahtbot with memory created


In [18]:
from langchain_core.messages import HumanMessage,AIMessage
config={"configurable": {"thread_id":"user-123"}}

result=chatbot.invoke(
   { "response":[HumanMessage(content="My name is Alice and I love Python programming.")]},config=config
)

print(f"BOT: {result['response'][-1].content}")

result2=chatbot.invoke({"response":HumanMessage(content="What do i love")},config=config)

print(f"BOT: {result2['response'][-1].content}")

result3=chatbot.invoke({"response":HumanMessage(content="Make a Joke about it")},config=config)

print(f"BOT: {result3['response'][-1].content}")


BOT: Hi Alice! It’s great to meet you! Your enthusiasm for Python programming is clear. If you have any questions about Python, specific projects you’d like to discuss, or topics you want to learn more about, feel free to share!
BOT: You mentioned that you love Python programming! If there are other interests or hobbies you'd like to share or explore, feel free to let me know!
BOT: Sure! Here's a Python programming joke for you:

Why do Python programmers prefer dark mode?

Because light attracts bugs! 🐍💻

Hope that brings a smile! If you want to hear more, just let me know!
